# PROFILING: ImmuVis trained on Jackson, tested on Danenberg (cross-cohort virtual staining)

**What this run does:**
- Trains the **published ImmuVis code** from scratch on the Jackson train patients and tests it on the Danenberg test patients.
  - Code: Uchal et al. 2026, `anonymous.4open.science/r/ImmuVis-9173`, with `github.com/dav3794/multiplex-image-model` as a fallback.
  - Architecture: the default `train_masked_config.yaml`, i.e. **ImmuVis-Conv** (ConvNeXt marker-agnostic + pan-marker encoder, hyperkernels, heteroscedastic decoder).
- The public ImmuVis weights are not used. They were pretrained on IMC17M, which contains both cohorts.

**Channels:** the 14 channels both cohorts share (H3, DNA1, DNA2 + the 11 shared markers), each as its own marker token.
- At test time each marker is predicted from the other 13, the same inputs as every other model in the figure.

**Preprocessing (ImmuVis pipeline):**
- arcsinh(x/5) → Butterworth low-pass (cutoff 0.2) → clip at 2.3 → /2.3.
- 2.3 is the published Jackson-Basel clip limit. It is used for Danenberg too, so no Danenberg statistics are used (level A).

**Training:**
- **Loss and masking**, as published: random 128×128 crops with rotation and flips; channel-subset sampling (≥ 75%); up to 50% of channels fully masked; 60% spatial masking with 8-px patches; β-NLL loss (β = 0.5); AdamW (lr 5e-4 → 1e-5, cosine, 10% warm-up), weight decay 1e-4.
- **Changes from the published setup:**
  - fp16 autocast, because a T4 has no bf16;
  - a time budget sets the number of steps;
  - the checkpoint is chosen by leave-one-marker-out pixel r on Jackson val patients.
- **Patients:** the same Jackson train/val patients as the Jackson-trained per-cell models (replicate 0 of `train-test-grid`).

**Inference:** 128-px tiles with 32-px context, stitched. Scored exactly like `cross-cohort-metrics`.

**Two kernels:** `immuvis-jackson-train` trains and saves the checkpoint; `immuvis-jackson` loads it and scores Danenberg. (The first single-kernel run was killed by Kaggle after 5.5 h with no log, losing the checkpoint.)

In [ ]:
import os, glob, json, time, sys, subprocess, warnings, traceback, re, pickle, gc, random
import numpy as np, pandas as pd
import pyarrow.parquet as pq
import scipy.sparse as sp
from scipy.spatial import cKDTree
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LinearRegression
from joblib import Parallel, delayed
try:
    import lightgbm as lgb
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm'], check=True)
    import lightgbm as lgb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

IN = os.environ.get("IN_ROOT", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
SMOKE = os.environ.get("SMOKE") == "1"
SEED = 20261013
N_BOOT = 100 if SMOKE else 1000
N_BOOT_F1 = 50 if SMOKE else 200
N_SHIFT = 5
MIN_CELLS = 200
COF = 5.0
RING_R = 30.0
N_FIT = 15_000 if SMOKE else 100_000          # cells per cross-fit half (R5) and per classifier
N_FIT_FINAL = 30_000 if SMOKE else 300_000    # Jackson cells for the baselines
LGB = dict(n_estimators=40 if SMOKE else 250, learning_rate=0.06, num_leaves=31, min_child_samples=50,
           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, n_jobs=os.cpu_count(), verbose=-1)
ARMS = ["residual", "gamma", "immuvis"]
NUC = ["histone_h3", "dna1", "dna2"]
# Image channel orders (as in AbundanceVsStructure). Danenberg index 12 'c_erb_b_2_her2' is clone 3B5.
D_CHAN = ['histone_h3', 'sma', 'cytokeratin_5', 'cd38', 'hla_dr', 'cytokeratin_8_18', 'cd15', 'fsp1', 'cd163',
          'cd278_icos', 'cd134', 'cd68', 'c_erb_b_2_her2', 'cd3', 'podoplanin', 'cd11c', 'cd279_pd_1', 'gitr_tnfrsf18',
          'cd16', 'c_erb_b_2_her2_d8f12', 'cd45ra', 'beta_2_microglobulin', 'cd45', 'foxp3', 'cd20',
          'estrogen_receptor_alpha', 'cd8a', 'cd57', 'ki_67', 'cd140b_pdgf_receptor_beta', 'caveolin_1', 'cd4',
          'cd31_v_wf', 'cxcl12_sdf_1', 'hla_abc', 'pan_cytokeratin', 'cleaved_caspase3', 'dna1', 'dna2']
J_CHAN = ['histone_h3', 'histone_h3_trimethylate', 'cytokeratin_5', 'fibronectin', 'cytokeratin_19', 'cytokeratin_8_18',
          'twist', 'cd68', 'keratin_14_krt14', 'sma', 'vimentin', 'c_myc', 'c_erb_b_2_her2', 'cd3', 'histone_h3_phospho',
          'slug', 'estrogen_receptor_alpha', 'progesterone_receptor_a_b', 'p53', 'cd44', 'cd45', 'gata3', 'cd20',
          'carbonic_anhydrase_ix', 'e_cadherin_p_cadherin', 'ki_67', 'egfr', 's6', 'v_wf', 'm_tor', 'cytokeratin_7',
          'pan_cytokeratin', 'cleaved_parp', 'dna1', 'dna2']
D_TABLE = {"c_erb_b_2_her2": "c_erb_b_2_her2_3b5"}       # image name -> Danenberg cell-table column
MARKERS = sorted(m for m in set(D_CHAN) & set(J_CHAN) if m not in NUC)    # = task_names in the training code
assert len(MARKERS) == 11, MARKERS
STROMAL_KEYS = ("fibro", "endothel", "stroma", "pdpn", "myofib", "vessel", "vwf")
T0 = time.time()

def tick(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

def g(pattern):
    return sorted(glob.glob(f"{IN}/**/{pattern}", recursive=True))

def near(name, anchor):
    hits = g(name)
    assert hits, f"missing input {name}"
    return max(hits, key=lambda c: len(os.path.commonpath([c, anchor])))

def guard(name):
    """Run an analysis block; on failure print the traceback and carry on, so the kernel ends COMPLETE."""
    def deco(fn):
        try:
            return fn()
        except Exception:
            print(f"!!! block {name} failed:\n{traceback.format_exc()}", flush=True)
            FAILED.append(name)
            return None
    return deco
FAILED = []

## Cell tables (Danenberg cells to analyse; Jackson cells for the baselines)

In [ ]:
P = {}
for p in g("cells.parquet"):
    key = "danenberg" if "metabric_id" in pq.read_schema(p).names else "jackson"
    P[key] = dict(cells=p, contacts=near("cell_neighbours.parquet", p), report=near("prep_report.json", p))
P["danenberg"].update(clinical=near("clinical.parquet", P["danenberg"]["cells"]),
                      split=near("danenberg_patient_split.csv", P["danenberg"]["cells"]))
print(json.dumps(P, indent=1))

def load_table(key):
    cells = pd.read_parquet(P[key]["cells"])
    if key == "danenberg":
        cells = cells[(cells.is_tumour == 1) & (cells.is_hot_aggregate == 0)]
        pid = "metabric_id"
    else:
        cells = cells[cells.is_tumour == 1]
        pid = "patient_id"
    n_per = cells.groupby("image_name").size()
    cells = cells[cells.image_name.isin(n_per.index[n_per >= MIN_CELLS])]
    cells = cells.sort_values(["image_name", "object_number"]).reset_index(drop=True)
    img_code, img_names = pd.factorize(cells.image_name, sort=True)
    starts = np.r_[0, np.flatnonzero(np.diff(img_code)) + 1, len(cells)]
    loc = pd.Series(np.arange(len(cells)), index=pd.MultiIndex.from_arrays(
        [cells.image_number.values, cells.object_number.values]))
    con = pd.read_parquet(P[key]["contacts"])
    ca = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["from"].values])).values
    cb = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["to"].values])).values
    ok = ~(np.isnan(ca) | np.isnan(cb))
    ca, cb = ca[ok].astype(np.int64), cb[ok].astype(np.int64)
    ca, cb = ca[ca != cb], cb[ca != cb]
    if key == "danenberg":
        epi_frac = cells.groupby("cell_phenotype").is_epithelial.mean()
        bmap = {p: "epithelial" if epi_frac[p] > 0.5 else
                "stromal" if any(s in p.lower() for s in STROMAL_KEYS) else "immune" for p in epi_frac.index}
        broad = cells.cell_phenotype.map(bmap).astype(str).values
        epi = cells.is_epithelial.values == 1
    else:
        broad = cells.broad_class.astype(str).values
        epi = broad == "epithelial"
    return dict(key=key, cells=cells, img=img_code, img_names=np.asarray(img_names), starts=starts,
                patient=cells[pid].astype(str).values, xy=cells[["x", "y"]].values.astype(np.float64),
                ca=ca, cb=cb, broad=broad, epi=epi, phen=cells.cell_phenotype.astype(str).values,
                log_area=np.log(np.maximum(cells["area"].values, 1)).astype(np.float32))

DD, DJ = load_table("danenberg"), load_table("jackson")
N = len(DD["cells"])
split = pd.read_csv(P["danenberg"]["split"]).set_index("metabric_id").split
DD["split"] = pd.Series(DD["patient"]).map(split).fillna("none").values
clin = pd.read_parquet(P["danenberg"]["clinical"]).replace({"nan": np.nan})
clin = clin.assign(pid=clin.metabric_id.astype(str)).groupby("pid").first()
print(f"Danenberg: {N:,} cells, {len(DD['img_names'])} images, {len(set(DD['patient']))} patients; "
      f"split cells {pd.Series(DD['split']).value_counts().to_dict()}")
print(f"Jackson: {len(DJ['cells']):,} cells, {len(DJ['img_names'])} images")
pats, pcode = np.unique(DD["patient"], return_inverse=True)
W = np.random.default_rng(SEED).multinomial(len(pats), np.full(len(pats), 1 / len(pats)), size=N_BOOT).astype(np.float64)
tick("tables loaded")

def contact_matrix(D):
    n = len(D["cells"])
    C = sp.coo_matrix((np.ones(len(D["ca"]), np.float32), (D["ca"], D["cb"])), shape=(n, n)).tocsr()
    C = (C + C.T).tocsr()
    C.data[:] = 1.0
    return C

def ring_means(X, D, C):
    """Mean of X over non-touching cells within RING_R of the centroid, and their count."""
    n, p = X.shape
    Mr = np.full((n, p), np.nan, np.float32)
    nr = np.zeros(n, np.float32)
    for a, b in zip(D["starts"][:-1], D["starts"][1:]):
        m = b - a
        pr = cKDTree(D["xy"][a:b]).query_pairs(RING_R, output_type="ndarray")
        R = sp.coo_matrix((np.ones(len(pr), np.float32), (pr[:, 0], pr[:, 1])), shape=(m, m)).tocsr()
        R = (R + R.T).tocsr()
        Ci = C[a:b, a:b].tocsr()
        R = (R - R.multiply(Ci)).tocsr()
        R.eliminate_zeros()
        dr = np.asarray(R.sum(1)).ravel()
        Mr[a:b] = np.where(dr[:, None] > 0, (R @ X[a:b]) / np.maximum(dr, 1)[:, None], np.nan)
        nr[a:b] = dr
    return Mr, nr

## Danenberg test set, truth maps and scoring (shared with `cross-cohort-metrics`)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imagecodecs==2024.9.22"], check=True)
import tifffile
import torch
dev = "cuda" if torch.cuda.is_available() else "cpu"
N_GPU = torch.cuda.device_count()

# truth cell means (target space) from kill-test, in cell-table order
KT = pd.read_parquet(g("percell_truth_pred.parquet")[0], columns=["image_name", "object_number"] + [f"truth__{m}" for m in MARKERS])
assert (KT.image_name.values == DD["cells"].image_name.values).all() and \
       (KT.object_number.values == DD["cells"].object_number.values).all(), "kill-test rows do not match the cell table"
TRUTH = {m: KT[f"truth__{m}"].values.astype(np.float32) for m in MARKERS}
del KT

tifs = g("*.tiff")
D_IMG = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs
         if os.path.basename(p).startswith("MB_") and "/masks/" not in p.replace("\\", "/")}
D_MASK = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs
          if os.path.basename(p).startswith("MB_") and "/masks/" in p.replace("\\", "/")}
J_IMG = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs
         if not os.path.basename(p).startswith("MB_") and "/masks/" not in p.replace("\\", "/")
         and "jackson" in p.lower()}
IMGS = list(DD["img_names"])
OBJ = DD["cells"].object_number.values.astype(np.int64)
ST = DD["starts"]
D_TE = np.flatnonzero(DD["split"] == "test")
IMG_TE = DD["img"][D_TE]
TEST_IMGS = np.unique(IMG_TE)
dix = {m: i for i, m in enumerate(D_CHAN)}
jix = {m: i for i, m in enumerate(J_CHAN)}
CACHE = os.path.dirname(g("channel_preproc_residual.pkl")[0])
_, TNS = pickle.load(open(f"{CACHE}/channel_preproc_residual.pkl", "rb"))
PIXK = pd.read_csv(g("pixel_sums_per_image.csv")[0])
PIXK = PIXK[PIXK.arm == "residual"].set_index(["marker", "image"])
print(f"Danenberg test: {len(D_TE):,} cells in {len(TEST_IMGS)} images; Jackson tiffs found: {len(J_IMG)}")

# Jackson split: replicate 0 of train-test-grid (same patients as the Jackson-trained per-cell models)
jp, jcode = np.unique(DJ["patient"], return_inverse=True)
_u = np.random.default_rng(SEED + 11 + 0).random(len(jp))
J_SPLIT = np.where(_u < 0.7, "train", np.where(_u < 0.8, "val", "test"))[jcode]
J_IMG_SPLIT = pd.Series(J_SPLIT, index=DJ["cells"].image_name.values).groupby(level=0).first()
J_TRAIN = sorted(s for s in J_IMG_SPLIT.index[J_IMG_SPLIT == "train"] if s in J_IMG)
J_VAL = sorted(s for s in J_IMG_SPLIT.index[J_IMG_SPLIT == "val"] if s in J_IMG)
print(f"Jackson images: train {len(J_TRAIN)}, val {len(J_VAL)} (tumour images of train / val patients)")

def read_hwc(path, n_chan):
    im = tifffile.imread(path)
    if im.ndim == 3 and im.shape[0] == n_chan and im.shape[-1] != n_chan:
        im = np.moveaxis(im, 0, -1)
    assert im.shape[-1] == n_chan, (path, im.shape)
    return im.astype(np.float32)

def truth_map(im_hwc, m):
    """The image models' target space: clip(arcsinh(x/5) / p99_Jackson, 0, 1) (p01 = 0)."""
    p01, p99 = TNS[jix[m]]
    return np.clip((np.arcsinh(im_hwc[..., dix[m]] / COF) - p01) / (p99 - p01 + 1e-8), 0., 1.)

def cell_index(i):
    """Pixel -> row in the test arrays, for test image i."""
    m = tifffile.imread(D_MASK[IMGS[i]]).astype(np.int64)
    a, b = ST[i], ST[i + 1]
    lut = np.full(max(int(m.max()), int(OBJ[a:b].max())) + 1, -1, np.int64)
    lut[OBJ[a:b]] = np.searchsorted(D_TE, np.arange(a, b))
    return lut[m]

def r_from(n, Sg, Sp, Sgg, Spp, Sgp):
    with np.errstate(invalid="ignore", divide="ignore"):
        return (Sgp - Sg * Sp / n) / np.sqrt((Sgg - Sg ** 2 / n) * (Spp - Sp ** 2 / n))

def r4_point(a, b):
    ok = np.isfinite(a) & np.isfinite(b)
    a, b, im = a[ok].astype(np.float64), b[ok].astype(np.float64), IMG_TE[ok]
    k = int(im.max()) + 1
    cnt = np.maximum(np.bincount(im, minlength=k), 1)
    a = a - (np.bincount(im, a, k) / cnt)[im]
    b = b - (np.bincount(im, b, k) / cnt)[im]
    return float((a * b).sum() / np.sqrt((a * a).sum() * (b * b).sum()))

def score_model(MODEL, predict_image):
    """predict_image(im_hwc) -> {marker: prediction map (H, W)}. Writes pixel sums and per-cell predictions."""
    rows = []
    CELLP = {m: np.full(len(D_TE), np.nan, np.float32) for m in MARKERS}
    for j, i in enumerate(TEST_IMGS):
        s = IMGS[i]
        im = read_hwc(D_IMG[s], len(D_CHAN))
        preds = predict_image(im)
        loc = cell_index(i)
        v = loc >= 0
        for m in MARKERS:
            G = truth_map(im, m).astype(np.float64).ravel()
            Pm = preds[m].astype(np.float64)
            assert Pm.shape == im.shape[:2], (s, m, Pm.shape)
            Pf = Pm.ravel()
            row = dict(model=MODEL, marker=m, image=s, n=G.size, Sg=G.sum(), Sgg=G @ G, Sp=Pf.sum(), Spp=Pf @ Pf, Sgp=G @ Pf)
            if j < 3:   # the truth map must equal kill-test's
                k_ = PIXK.loc[(m, s)]
                assert abs(row["Sg"] - k_.Sg) <= 1e-4 * max(1.0, abs(k_.Sg)) and row["n"] == k_.n, (s, m, row["Sg"], k_.Sg)
            rows.append(row)
            cid = loc[v]
            cnt = np.bincount(cid, minlength=len(D_TE))
            sm = np.bincount(cid, Pm[v], len(D_TE))
            got = cnt > 0
            CELLP[m][got] = (sm[got] / cnt[got]).astype(np.float32)
        if j % 20 == 0:
            tick(f"{MODEL}: scored {j + 1}/{len(TEST_IMGS)} test images")
    S = pd.DataFrame(rows)
    tag = MODEL.lower().replace(" ", "-")
    S.to_csv(f"{OUT}/pixel_sums_test_{tag}.csv", index=False)
    np.savez_compressed(f"{OUT}/percell_pred_{tag}.npz", image=np.asarray(IMGS)[IMG_TE], object_number=OBJ[D_TE],
                        **{f"pred__{MODEL}__{m}": CELLP[m] for m in MARKERS})
    # point summaries (CIs are computed locally with the other models, same bootstrap draws)
    out = []
    for m in MARKERS:
        d = S[S.marker == m]
        ri = r_from(d.n.values, d.Sg.values, d.Sp.values, d.Sgg.values, d.Spp.values, d.Sgp.values)
        out.append(dict(marker=m, pooled=float(r_from(*(d[c].sum() for c in ["n", "Sg", "Sp", "Sgg", "Spp", "Sgp"]))),
                        per_image=float(np.nanmean(ri)), R4=r4_point(TRUTH[m][D_TE], CELLP[m])))
    out = pd.DataFrame(out)
    out.to_csv(f"{OUT}/metrics_point_{tag}.csv", index=False)
    print(out.round(3).to_string(index=False))
    print("mean over 10 markers (ERα excluded):",
          out[out.marker != "estrogen_receptor_alpha"][["pooled", "per_image", "R4"]].mean().round(3).to_dict())
    return S, CELLP

## ImmuVis code and model configuration

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "comet_ml", "ruamel.yaml"], check=True)
import zipfile, urllib.request, copy
from torch.amp import autocast, GradScaler
import torchvision.transforms.functional as TF
from skimage.filters import butterworth

CODE = "/tmp/immuvis_code"
def fetch_code():
    try:
        urllib.request.urlretrieve("https://anonymous.4open.science/api/repo/ImmuVis-9173/zip", "/tmp/immuvis.zip")
        zipfile.ZipFile("/tmp/immuvis.zip").extractall(CODE)
        return "anonymous.4open.science/r/ImmuVis-9173"
    except Exception as e:
        print("anonymous release not available:", e)
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/dav3794/multiplex-image-model.git", CODE], check=True)
        return "github.com/dav3794/multiplex-image-model"
CODE_SOURCE = fetch_code()
root = CODE if os.path.isdir(f"{CODE}/multiplex_model") else os.path.dirname(glob.glob(f"{CODE}/**/multiplex_model", recursive=True)[0])
sys.path.insert(0, root)
from multiplex_model.modules import MultiplexAutoencoder
from multiplex_model.utils import (EncoderConfig, DecoderConfig, apply_channel_masking, apply_spatial_masking,
                                   ClampWithGrad, get_scheduler_with_warmup)
from multiplex_model.losses import beta_nll_loss
print("ImmuVis code from", CODE_SOURCE)
# cuDNN on a T4 ran this model at ~6 s/step and leaked ~50 MB host RAM per step (immuvis-profile);
# with cuDNN off: 0.58 s/step and flat RAM
torch.backends.cudnn.enabled = False

CH = NUC + MARKERS                         # 14 shared channels = 14 marker tokens
IDS = torch.arange(len(CH))
CLIP = 2.3
CROP, BATCH = 128, 16
TRAIN_MIN = float(os.environ.get("TRAIN_MIN", 180))
EVAL_EVERY = 500

def immuvis_pre(im_hwc, names, idx):
    x = np.arcsinh(np.stack([im_hwc[..., idx[c]] for c in names]) / COF)
    x = np.stack([butterworth(c, cutoff_frequency_ratio=0.2, high_pass=False) for c in x])
    return (np.clip(x, 0, CLIP) / CLIP).astype(np.float16)

ENC = EncoderConfig(ma_layers_blocks=[4], ma_embedding_dims=[16], pm_layers_blocks=[4, 4, 4],
                    pm_embedding_dims=[192, 384, 768],
                    hyperkernel=dict(kernel_size=1, padding=0, stride=1, use_bias=True)).model_dump()
DEC = DecoderConfig(decoded_embed_dim=512, num_blocks=1,
                    hyperkernel=dict(kernel_size=1, padding=0, stride=1, use_bias=True)).model_dump()

## Profile: speed per configuration, top GPU ops, validation, checkpoint, memory over 20 min

In [ ]:
import psutil, shutil
# Profile v2. v1 showed host RAM growing ~50 MB per step without a plateau and ~8 s/step, mostly CPU time inside
# convolution calls. Hypothesis: cuDNN plans are rebuilt and cached for every new tensor shape, and the shapes change
# every step because the channel masking keeps a random number of channels. Each fix runs in its own process.

def prep_jackson(s):
    return immuvis_pre(read_hwc(J_IMG[s], len(J_CHAN)), CH, jix)

JTR = Parallel(n_jobs=4)(delayed(prep_jackson)(s) for s in J_TRAIN[:60])
from joblib.externals.loky import get_reusable_executor
get_reusable_executor().shutdown(wait=True)
np.savez("/tmp/jtr.npz", *JTR)
del JTR
gc.collect()
tick("60 Jackson train images preprocessed and saved")

SCRIPT = r"""
import os, sys, json, time, gc
cfg = json.loads(sys.argv[1])
if cfg.get("lru"):
    os.environ["TORCH_CUDNN_V8_API_LRU_CACHE_LIMIT"] = str(cfg["lru"])
import numpy as np, torch, psutil
import torchvision.transforms.functional as TF
from torch.amp import autocast, GradScaler
sys.path.insert(0, cfg["root"])
from multiplex_model.modules import MultiplexAutoencoder
from multiplex_model.utils import EncoderConfig, DecoderConfig, apply_channel_masking, apply_spatial_masking, ClampWithGrad
from multiplex_model.losses import beta_nll_loss
torch.backends.cudnn.enabled = cfg["cudnn"]
torch.backends.cudnn.benchmark = cfg["bench"]
dev = "cuda"
Z = np.load("/tmp/jtr.npz")
JTR = [Z[k] for k in Z.files]
C = JTR[0].shape[0]
IDS = torch.arange(C)
ENC = EncoderConfig(ma_layers_blocks=[4], ma_embedding_dims=[16], pm_layers_blocks=[4, 4, 4], pm_embedding_dims=[192, 384, 768],
                    hyperkernel=dict(kernel_size=1, padding=0, stride=1, use_bias=True)).model_dump()
DEC = DecoderConfig(decoded_embed_dim=512, num_blocks=1, hyperkernel=dict(kernel_size=1, padding=0, stride=1, use_bias=True)).model_dump()
torch.manual_seed(0)
net = MultiplexAutoencoder(num_channels=C, encoder_config=ENC, decoder_config=DEC).to(dev)
model = torch.nn.DataParallel(net) if cfg["dp"] else net
opt = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
scaler = GradScaler()
rng = np.random.default_rng(0)
CROP, B = 128, cfg["batch"]
WIN = int(np.ceil(CROP * np.sqrt(2))) + 2

def batch():
    out = []
    for _ in range(B):
        x = JTR[rng.integers(len(JTR))]
        H, W = x.shape[1:]
        if H < WIN or W < WIN:
            x = np.pad(x, ((0, 0), (0, max(0, WIN - H)), (0, max(0, WIN - W))))
            H, W = x.shape[1:]
        y0, x0 = rng.integers(0, H - WIN + 1), rng.integers(0, W - WIN + 1)
        out.append(x[:, y0:y0 + WIN, x0:x0 + WIN])
    t = torch.from_numpy(np.stack(out).astype(np.float32)).to(dev)
    res = []
    for b in range(B):
        r = TF.center_crop(TF.rotate(t[b], float(rng.uniform(-180, 180)), interpolation=TF.InterpolationMode.BILINEAR), [CROP, CROP])
        res.append(torch.flip(r, dims=[-1]) if rng.random() < 0.5 else r)
    return torch.stack(res)

def masking(img, ids):
    if cfg["fixed"]:                    # fixed shapes: always 12 sampled channels, 4 of them fully masked
        sub = torch.stack([torch.randperm(C)[:12] for _ in range(len(img))])
        img = torch.stack([img[b, sub[b]] for b in range(len(img))])
        ids = torch.stack([ids[b, sub[b]] for b in range(len(img))])
        keep = torch.stack([torch.randperm(12)[4:] for _ in range(len(img))])
        masked = torch.stack([img[b, keep[b]] for b in range(len(img))])
        act = torch.stack([ids[b, keep[b]] for b in range(len(img))])
        return img, ids, masked, act
    return apply_channel_masking(img, ids, 0.75, 0.5, apply_channel_subset_sampling=True)

trace, t_start = [], time.time()
for step in range(1, cfg["steps"] + 1):
    t0 = time.time()
    img = batch()
    ids = IDS.to(dev).expand(len(img), -1)
    img, ids, masked, act = masking(img, ids)
    masked, _ = apply_spatial_masking(masked, 0.6, 8)
    with autocast(device_type="cuda", dtype=torch.float16):
        out = model(masked, act, ids)["output"]
    mi, logvar = out.float().unbind(dim=-1)
    loss = beta_nll_loss(img, torch.sigmoid(mi), ClampWithGrad.apply(logvar, -15.0, 15.0), beta=0.5)
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    scaler.step(opt)
    scaler.update()
    opt.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    if step % 10 == 0:
        trace.append(dict(step=step, s_per_step=time.time() - t0, elapsed=time.time() - t_start,
                          rss_gb=psutil.Process().memory_info().rss / 2**30,
                          gpu_peak_gb=torch.cuda.max_memory_allocated() / 2**30, loss=float(loss.item())))
        print(json.dumps(trace[-1]), flush=True)
    if time.time() - t_start > cfg["max_s"]:
        break
print("RESULT " + json.dumps(dict(cfg=cfg["name"], steps=step, s_per_step=(time.time() - t_start) / step,
                                  rss_start=trace[0]["rss_gb"] if trace else None, rss_end=trace[-1]["rss_gb"] if trace else None,
                                  gpu_peak_gb=torch.cuda.max_memory_allocated() / 2**30)), flush=True)
"""
open("/tmp/prof.py", "w").write(SCRIPT)
base = dict(root=root, steps=120, max_s=360, batch=16, dp=True, cudnn=True, bench=False, lru=0, fixed=False)
CONFIGS = [dict(base, name="as before (cuDNN on, 2 GPUs)"),
           dict(base, name="cuDNN plan cache limited to 50", lru=50),
           dict(base, name="cuDNN off", cudnn=False),
           dict(base, name="fixed channel counts (12 sampled, 4 masked), cuDNN benchmark", fixed=True, bench=True),
           dict(base, name="fixed channel counts, cuDNN off", fixed=True, cudnn=False),
           dict(base, name="cuDNN off, 1 GPU", cudnn=False, dp=False)]
RES = []
for c in CONFIGS:
    t0 = time.time()
    r = subprocess.run([sys.executable, "/tmp/prof.py", json.dumps(c)], capture_output=True, text=True)
    lines = r.stdout.strip().splitlines()
    print(f"--- {c['name']} ({(time.time() - t0) / 60:.1f} min, exit {r.returncode})")
    print("\n".join(lines[-4:]))
    if r.returncode != 0:
        print(r.stderr[-1500:])
    res = [json.loads(l[7:]) for l in lines if l.startswith("RESULT ")]
    RES.append(res[0] if res else dict(cfg=c["name"], error=r.stderr[-300:]))
    pd.DataFrame(RES).to_csv(f"{OUT}/immuvis_profile_v2.csv", index=False)
print(pd.DataFrame(RES).round(3).to_string(index=False))
tick("profile v2 done")